# Day 2 Starter Kit: Complete the Grounding (no Neo4j version)

**Your dataset: Employee, contractor & manager (code HR).** A company's people-operations log: recruitment, onboarding, payroll, contractor billing, reviews, promotions, leave and workplace services.

**The idea.** Retrieval finds records that *sound* similar. It cannot tell whether two records describe the *same event*. The graph can, **if it knows which parts of a record identify an event.** That knowledge is a list of regular expressions, and writing it well is your job.

**What is given and working:** retrieval, the baseline RAG, graph building (lookup tables) from a *pattern registry*, and the grounded answer step. The registry has **one pattern** (Employee ID).

**What you do**
0. **Plug in your own LLM.** Add your key from any provider and, if needed, edit the one `call_llm` function so the notebook runs on it. Nothing else in the notebook touches the provider.
1. Explore the data and decide which other patterns identify a single event (**Kind 1**) and which are incidental fields reused across unrelated events (**Kind 2**: a room number, an amount, a semester...).
2. Add your patterns to the registry, each with a reason. List the lookalikes you *rejected*, and why.
3. Run the public question pack against baseline and grounded. Explain every outcome.
4. Say where the baseline fails and your graph works, and **one failure you still have.**

**No database needed.** This version keeps the graph in plain Python lookup tables, already built for you, so there is no Neo4j to set up. Your work is the registry. The judging is identical to the Neo4j version.

**What you submit:** this notebook, run top to bottom on a fresh runtime. The judges read your registry, your outcomes table and your remaining-failure note, then run a hidden question pack against your graph.

> Linking on the wrong thing is as bad as linking on nothing: a pattern that links unrelated records makes the system *invent* connections. Add a pattern only if you can say why two records sharing it are the same event.

## 1. Setup

In [ ]:
%pip install -q sentence-transformers hnswlib openai python-dotenv

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready")

### Your LLM provider

Bring your own key (any provider) and add it to Colab's Secrets manager (padlock icon) as `LLM_API_KEY`. Every LLM call in this notebook goes through the single function `call_llm` below.

Run the cell: it should print a one-word reply. If it errors, fix this cell before going on.

In [ ]:
# ---- YOUR LLM PROVIDER (your own key, any provider) ----
# Add your key to Colab Secrets as LLM_API_KEY. The whole notebook talks to the model through
# ONE function, call_llm(prompt) -> str, so this is the only cell you change for a new provider.
# The default below works for OpenAI and for any provider with an OpenAI-compatible endpoint:
# set LLM_MODEL (and LLM_BASE_URL if it is not OpenAI) from your provider's documentation.
# A provider with its own SDK: keep the name and signature of call_llm, rewrite the body.
from openai import OpenAI

LLM_MODEL = "gpt-4o-mini"
LLM_BASE_URL = None  # None = OpenAI. Otherwise your provider's OpenAI-compatible URL

client = OpenAI(api_key=get_secret("LLM_API_KEY"), base_url=LLM_BASE_URL)

def call_llm(prompt):
    """Sends one prompt, returns the model's reply as plain text.
    Keep temperature=0: otherwise the same question can give a different answer on each run."""
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

print("LLM check:", call_llm("Reply with the single word: ready"))

### No database needed: the graph lives in lookup tables

This version of the starter holds the graph in plain Python lookup tables instead of a Neo4j database, so there is nothing to sign up for or connect. Section 7 builds the tables from your registry and `graph_check` in section 8 walks them. They answer the same question a graph database would: *is there a path of shared entities between these two records?*

## 2. The dataset: Employee, contractor & manager

141 records in 12 categories. A company's people-operations log: recruitment, onboarding, payroll, contractor billing, reviews, promotions, leave and workplace services. Four pairs of categories deliberately share vocabulary. Most records stand alone; some carry an identifier or a code.

In [ ]:
DATASET_ID = "HR"
DATASET_NAME = "Employee, contractor & manager"

rec_categories = {
    'Recruitment': [
        'A job posting for a data analyst role was published on the careers page',
        'Recruiters screened a batch of applications for the sales team',
        'An interview panel was scheduled for a senior engineer candidate',
        'A candidate declined an offer after a counter from another company',
        'Reference checks were completed for a finance candidate',
        'The hiring freeze was lifted for two customer support roles',
        'A campus hiring drive was planned for the next graduating class',
        'An offer letter was revised after the salary band was updated',
        'A recruiter reopened a role that had stayed unfilled for two months',
        'A Grade L3 role was opened for a backend developer',
        'Hiring for the support team was paused until the Q3 appraisal closed',
        'Contractor K230 was engaged through a staffing agency for six months',
    ],
    'Onboarding': [
        'A new hire orientation was held for the incoming batch',
        'IT prepared laptops and access cards for the joiners',
        'A buddy was assigned to a new team member',
        'Onboarding paperwork was completed before the first day',
        'A welcome kit was sent to a remote joiner',
        'The induction schedule was shortened for experienced hires',
        "A new joiner's security training was marked pending",
        'Probation goals were shared with the new team members',
        'The onboarding survey showed a gap in first-week tool access',
        'Employee E1021 joined the finance team through an employee referral',
        'Contractor K214 was issued a laptop and access card before starting',
    ],
    'Payroll': [
        'Monthly payroll was processed and salary slips were released',
        'A tax declaration deadline was announced to all staff',
        'Payroll corrected an overpayment from the previous cycle',
        'A bonus payout was scheduled with the next salary run',
        'Provident fund contributions were reconciled by payroll',
        'Payroll closed the cycle after a final attendance check',
        'A salary advance request was approved by finance',
        'Reimbursement claims were added to the payroll file',
        'A bank file for salary transfers was validated',
        "Employee E1034's new salary was reflected in the payroll run",
        "Employee E1021's relocation allowance of 3000 dollars was approved",
    ],
    'Contractor_Billing': [
        'A contractor invoice was approved after timesheet verification',
        'Contractor payments were released on the net payment cycle',
        'A staffing agency invoice was disputed for extra hours',
        'Withholding tax was applied to a contractor payment',
        "A contractor's rate card was updated for the new year",
        'Procurement raised a purchase order for outside developers',
        'A late timesheet delayed a contractor payment',
        'A vendor invoice for consulting hours was queued for review',
        'Contractor expenses were reimbursed against receipts',
        "Contractor K214's invoice was approved after timesheet verification",
        "Contractor K230's rate card was updated after the engagement was extended",
        "A reimbursement of 800 dollars was approved for a contractor's travel",
        'A refund of 3000 dollars was issued to a vendor after a billing error',
    ],
    'Performance_Reviews': [
        'Mid-year review forms were opened for managers',
        'A calibration meeting was held for the engineering teams',
        'An employee requested feedback after a rating dispute',
        'Review ratings were locked after calibration',
        'A performance improvement plan was drafted for a struggling team member',
        'Peer feedback was collected for the product group',
        'Managers were reminded to complete review comments',
        'A self-assessment template was updated for the year',
        'Review outcomes were communicated in one-to-one meetings',
        'Employee E1021 received a strong mid-year rating after calibration',
        'Manager M09 completed review comments for all direct reports',
        'Employee E1066 was placed on a performance improvement plan',
        'Q3 appraisal ratings were frozen for the engineering teams',
    ],
    'Promotions': [
        'A promotion list was submitted for leadership approval',
        'Compensation bands were reviewed for the technology roles',
        'A salary revision letter was issued after a promotion',
        'A lateral move was approved between two departments',
        'The promotion committee deferred a case for more evidence',
        'An increment budget was allocated to each department',
        'A retention bonus was offered to a critical employee',
        'A role change was announced with a new reporting line',
        'Equity grants were scheduled for the next vesting date',
        'Employee E1034 was recommended for promotion by the committee',
        'Manager M05 submitted the promotion list for the design team',
        'Compensation bands for Grade L3 were revised for the next year',
    ],
    'Leave_Attendance': [
        'A parental leave request was approved by the manager',
        'Attendance records were corrected after a system outage',
        'A long weekend holiday was announced for all offices',
        'An employee applied for unpaid leave for personal reasons',
        'Leave balances were carried forward into the new year',
        'A comp-off request was approved for weekend work',
        'Biometric attendance was paused during maintenance',
        'Sick leave certificates were collected by HR',
        'A leave policy reminder was sent before the festival season',
        "Employee E1052's parental leave request was approved",
    ],
    'Workplace_Facilities': [
        'A desk allocation plan was published for the new office',
        'The cafeteria menu was updated after employee feedback',
        'An air conditioning fault was reported in the meeting room',
        'Parking slots were reassigned for the new quarter',
        'Access cards were replaced for two departments',
        'The security desk logged a visitor badge mismatch',
        'A hybrid seating system was rolled out across offices',
        'Fire drill was conducted at the main office',
        'The shuttle service timing was changed for night shifts',
        'Floor 5 was allocated to the product team in the new seating plan',
        'A water leak on Floor 5 was reported to the facilities desk',
        'Employee E1047 reported noisy construction on Floor 8',
        'Employee E1066 requested a quiet desk on Floor 8',
    ],
    'Training': [
        'A leadership workshop was announced for new managers',
        'An online course on data privacy was made mandatory',
        'A coding bootcamp was scheduled for junior engineers',
        'Training budget was approved for external certifications',
        'A language class was offered to relocating employees',
        'Safety training was completed by the plant staff',
        'A mentoring programme was launched for first-time managers',
        'Training attendance was recorded in the learning portal',
        'A skills gap survey was circulated to all departments',
        'Employee E1052 completed the data privacy course before going on leave',
        'Manager M09 attended the leadership workshop for new managers',
        'A training grant of 800 dollars was approved for an external certification',
    ],
    'Employee_Relations': [
        'A grievance was filed about unequal shift allocation',
        'Employee relations mediated a dispute between two colleagues',
        'A town hall was organised to discuss the new policy',
        'Anonymous feedback was reviewed by the culture committee',
        'A harassment awareness session was held for all teams',
        'A union representative met with HR about working hours',
        'The ethics hotline received a report that was routed for review',
        'An employee engagement survey was launched across offices',
        'A team outing was approved to improve morale',
        'Case HRC-3307 was opened for a grievance about shift allocation',
        'Case HRC-3307 was closed after mediation and a written resolution',
        'Manager M14 attended the town hall on the new policy',
    ],
    'Offboarding': [
        'An exit interview was scheduled for a departing employee',
        'Final settlement was prepared for a resigned employee',
        'Access was revoked on the last working day',
        'A notice period buyout was approved by HR',
        'Company assets were collected from a leaving employee',
        'A relieving letter was issued after clearance',
        'An alumni network invitation was sent to leavers',
        'Knowledge transfer sessions were planned before a resignation took effect',
        'A rehire request was reviewed against past exit notes',
        'Employee E1047 resigned and a final settlement was prepared',
    ],
    'Compliance': [
        'Annual code of conduct acknowledgement was collected',
        'A labour law update was shared with HR business partners',
        'An audit requested records of overtime approvals',
        'Data retention rules for personnel files were revised',
        'A background check vendor was changed after a contract review',
        'Mandatory POSH committee membership was refreshed',
        'Minimum wage changes were reviewed for contract staff',
        'A policy exception was documented and approved',
        'Statutory filings for the quarter were submitted on time',
        'Contractor K201 completed the code of conduct acknowledgement',
        'Case HRC-2218 was opened for a conflict of interest disclosure',
        'Q3 appraisal forms were sent to all managers',
    ],
}

rec_texts = []
rec_labels = []
for category, sentences in rec_categories.items():
    for s in sentences:
        rec_texts.append(s)
        rec_labels.append(category)

print(f"Dataset {DATASET_ID}: {len(rec_texts)} records, {len(rec_categories)} categories")

## 3. Retrieval (given: this part works)

In [ ]:
rec_embeddings = model.encode(rec_texts, show_progress_bar=True)
rec_embeddings = np.ascontiguousarray(rec_embeddings.astype('float32'))

dim = rec_embeddings.shape[1]
rec_index = hnswlib.Index(space='cosine', dim=dim)
rec_index.init_index(max_elements=len(rec_texts), ef_construction=200, M=16)
rec_index.add_items(rec_embeddings, np.arange(len(rec_texts)))
rec_index.set_ef(50)
print(f"Index built: {len(rec_texts)} records")


def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = rec_index.knn_query(q_emb, k=k)
    return [(int(idx), rec_texts[idx], rec_labels[idx]) for idx in labels[0]]

## 4. Baseline RAG (given: no grounding)

The plain Week 2 pipeline. It retrieves similar records and trusts the LLM to reason about whether they are connected.

In [ ]:
# The plain Week 2 prompt -- no grounding. temperature=0 so runs are repeatable.
BASELINE_PROMPT = """Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""

def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = BASELINE_PROMPT.format(context=context, question=question)
    return {"question": question, "retrieved": results, "answer": call_llm(prompt)}

## 5. Explore: what could identify one event?

Read the records below. For each string with digits, ask: *if I saw this same string in a different record, would it still point at the same event?*

In [ ]:
import re

def show_id_like_records(max_rows=80):
    """Prints every record containing a digit. Read them: which of these strings
    would still point at ONE specific event if you saw it in a different record?"""
    n = 0
    for i, t in enumerate(rec_texts):
        if re.search(r"\d", t):
            print(f"R{i:<3} {t}")
            n += 1
            if n >= max_rows:
                break
    print(f"\n{n} records shown")

show_id_like_records()

## 6. YOUR TASK: the pattern registry

Each entry becomes a graph node label. Two records that mention the *same identifier* get connected through that node. Edit this cell, then run it.

In [ ]:
import re

PATTERNS = [
    {
        "name": 'Employee',
        "regex": r"\bEmployee\s+(E\d{4})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Employee E1034',
        "why": 'An employee ID names one person: the promotion recommendation and the payroll update for E1034 are about the same employee.',
    },
    # YOUR PATTERNS GO HERE. One entry per pattern, same five fields as above.
    # Not sure a pattern is right? Test it first:
    #   re.findall(r"your regex", campus_texts[i])   or loop over all campus_texts.
    # {
    #     "name": "Xyz",                      # names the type of identifier
    #     "regex": r"...",                        # ONE capture group around the identifier
    #     "kind": 1,                              # 1 = names ONE event/person; 2 = incidental field
    #     "example": "...",                       # a string from the data it matches
    #     "why": "...",                           # why two records sharing it ARE the same event
    # },
]

REJECTED_CANDIDATES = [
    # Things that LOOK like identifiers but you decided NOT to link on. Credit is given for good reasons.
    # {"candidate": "...", "kind": 2, "why_rejected": "..."},
]

def validate_registry():
    """Checks that every pattern is complete and compiles. Run this before building the graph."""
    problems = []
    for p in PATTERNS:
        label = p.get("name", "?")
        for key in ("name", "regex", "kind", "example", "why"):
            if p.get(key) in (None, ""):
                problems.append(f"{label}: missing '{key}'")
        if p.get("name") and not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", p["name"]):
            problems.append(f"{label}: name must be letters/digits/underscore only")
        try:
            rx = re.compile(p.get("regex", ""))
            if rx.groups < 1:
                problems.append(f"{label}: regex needs ONE capture group around the identifier")
        except re.error as e:
            problems.append(f"{label}: regex does not compile ({e})")
        if p.get("kind") not in (1, 2):
            problems.append(f"{label}: kind must be 1 or 2")
    return problems

def show_register():
    print(f"{'Pattern':<10} {'Kind':<5} {'Example':<20} Why")
    for p in PATTERNS:
        print(f"{p['name']:<10} {p['kind']:<5} {p['example']:<20} {p['why']}")
    if REJECTED_CANDIDATES:
        print("\nConsidered but rejected:")
        for r in REJECTED_CANDIDATES:
            print(f"  - {r['candidate']} (Kind {r['kind']}): {r['why_rejected']}")

problems = validate_registry()
print("Registry problems:", problems if problems else "none")
show_register()

## 7. Build the lookup tables from your registry (given)

Re-run this cell every time you change the registry. Three tables hold the graph: `mentions` (one row per record-entity match) and the same rows indexed both ways (`entities_of`, `records_of`). The summary shows how many links each pattern created: a pattern with **0 shared identifiers** adds nothing, and one that links *many* records deserves suspicion.

In [ ]:
def extract_entities(text):
    """Applies every registered pattern. Returns [(pattern_name, identifier), ...]."""
    found = []
    for p in PATTERNS:
        for m in re.finditer(p["regex"], text):
            ent = (p["name"], m.group(1))
            if ent not in found:
                found.append(ent)
    return found

assert not validate_registry(), "Fix the registry problems above first"

# Table 1: one row per mention: (record_id, entity_type, entity_value)
mentions = []
for i, text in enumerate(rec_texts):
    for etype, value in extract_entities(text):
        mentions.append((f"R{i}", etype, value))

# Tables 2 and 3: the same rows indexed both ways, so we can walk record -> entity -> record
entities_of = {}   # "R12" -> {("Student", "S001"), ...}
records_of = {}    # ("Student", "S001") -> {"R12", "R57", ...}
for rec, etype, value in mentions:
    entities_of.setdefault(rec, set()).add((etype, value))
    records_of.setdefault((etype, value), set()).add(rec)

print(f"Lookup tables built from your registry: {len(mentions)} mention rows")
for name in sorted({m[1] for m in mentions}):
    ids = {}
    for rec, etype, value in mentions:
        if etype == name:
            ids[value] = ids.get(value, 0) + 1
    shared = sum(1 for c in ids.values() if c > 1)
    print(f"  {name:<10} {sum(ids.values()):>3} mentions | {len(ids):>3} distinct identifiers | {shared} shared by 2+ records (= links)")

In [ ]:
# What the lookup table looks like: the first 12 rows
print(f"{'record':<8}{'entity_type':<14}entity_value")
for row in mentions[:12]:
    print(f"{row[0]:<8}{row[1]:<14}{row[2]}")

## 8. Grounded answer (given)

`graph_check` walks the lookup tables to see whether two records are connected (and through which entity). The grounded step asks it about each pair of retrieved records, then hands the LLM those facts with record IDs. `temperature=0`, so repeated runs give the same answer.

In [ ]:
from collections import deque

def graph_check(id_a, id_b, max_hops=6):
    """Are two records connected? Returns {"disconnected": bool, "via": [...]}.
    Walks record -> entity -> record using the lookup tables; each step is one hop,
    and the walk stops after 6 hops."""
    start, goal = ("R", id_a), ("R", id_b)
    prev, queue = {start: None}, deque([(start, 0)])
    while queue:
        node, depth = queue.popleft()
        if node == goal:
            break
        if depth >= max_hops:
            continue
        if node[0] == "R":
            neighbours = [("E", etype, value) for etype, value in entities_of.get(node[1], ())]
        else:
            neighbours = [("R", rec) for rec in records_of[(node[1], node[2])]]
        for nb in neighbours:
            if nb not in prev:
                prev[nb] = node
                queue.append((nb, depth + 1))
    if goal not in prev:
        return {"disconnected": True, "via": None}
    path, node = [], goal
    while node is not None:
        path.append(node)
        node = prev[node]
    return {"disconnected": False, "via": [f"{n[1]} {n[2]}" for n in reversed(path) if n[0] == "E"]}


GROUNDED_PROMPT = """You answer questions about workforce operations (hiring, payroll, reviews, contractors, facilities) records. Use ONLY the records and graph facts below.

Rules:
1. The question's wording is a claim to check, NOT evidence. Do not accept a link just because the question assumes one.
2. Amounts, floor numbers, grades, appraisal periods, dates and similar shared values are NOT links.
3. The graph facts are ground truth about links. If the two records the question is about are NOT CONNECTED,
   say plainly that no recorded link exists between them. If they are CONNECTED, say so and name what they share.
4. If the records do not answer the question, say so. Cite record IDs like [R12].

Records:
{context}

Graph facts (checked in the database):
{graph_facts}

Question: {question}

Answer:"""


def grounded_answer(question, k=3):
    results = retrieve(question, k=k)
    ids = [f"R{idx}" for idx, _, _ in results]

    fact_lines, verified_connect, verified_disconnect = [], False, False
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            chk = graph_check(ids[i], ids[j])
            if chk is None:
                continue
            if chk["disconnected"]:
                verified_disconnect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: NOT CONNECTED -- no shared named entity links them.")
            else:
                verified_connect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: CONNECTED -- via {', '.join(chk['via'] or [])}.")

    context = "\n".join(f"- [R{idx}] {text}" for idx, text, _ in results)
    prompt = GROUNDED_PROMPT.format(context=context,
                                    graph_facts="\n".join(fact_lines) or "(no pairs to check)",
                                    question=question)
    return {"question": question, "retrieved": results, "graph_facts": fact_lines,
            "verified_connect": verified_connect, "verified_disconnect": verified_disconnect,
            "answer": call_llm(prompt)}

## 9. The public question pack

8 adversarial questions (the right answer is *no recorded connection*) and 3 positive controls (the right answer is *yes*). The judges will run a **different, hidden** pack of the same shapes, so fix your **registry**, not individual questions.

In [ ]:
PUBLIC_ADVERSARIAL = [{'id': 'P1',
  'question': 'Was the bonus payout released together with the contractor payments?',
  'trap_records': ['A bonus payout was scheduled with the next salary run',
                   'Contractor payments were released on the net payment cycle'],
  'correct_answer_must_state': 'No stated connection: a bonus payout and contractor payments are '
                               'separate records about pay.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (money)'},
 {'id': 'P2',
  'question': 'Is the revised offer letter the one whose onboarding paperwork was completed before '
              'the first day?',
  'trap_records': ['An offer letter was revised after the salary band was updated',
                   'Onboarding paperwork was completed before the first day'],
  'correct_answer_must_state': 'No evidence these records name the same person; they are separate '
                               'records.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (hiring)'},
 {'id': 'P3',
  'question': 'Did locking the review ratings cause the promotion committee to defer the case?',
  'trap_records': ['Review ratings were locked after calibration',
                   'The promotion committee deferred a case for more evidence'],
  'correct_answer_must_state': 'No stated connection: the deferral cites more evidence, and '
                               'nothing ties it to the ratings lock.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (review)'},
 {'id': 'P4',
  'question': 'Was the shuttle timing changed because of the long weekend holiday?',
  'trap_records': ['A long weekend holiday was announced for all offices',
                   'The shuttle service timing was changed for night shifts'],
  'correct_answer_must_state': 'No stated connection: the shuttle change concerns night shifts and '
                               'names no holiday.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (schedule)'},
 {'id': 'P5',
  'question': 'Did the new seating plan for the product team cause the water leak on Floor 5?',
  'trap_records': ['Floor 5 was allocated to the product team in the new seating plan',
                   'A water leak on Floor 5 was reported to the facilities desk'],
  'correct_answer_must_state': 'No stated connection: both mention Floor 5, but a floor is a place '
                               'shared by many events, not an event identifier.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: floor'},
 {'id': 'P6',
  'question': 'Was the Grade L3 role opened because the compensation bands for Grade L3 were '
              'revised?',
  'trap_records': ['A Grade L3 role was opened for a backend developer',
                   'Compensation bands for Grade L3 were revised for the next year'],
  'correct_answer_must_state': 'No stated connection: both mention Grade L3, a level shared by '
                               'many roles, not one event.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: grade'},
 {'id': 'P7',
  'question': "Was the 800 dollar training grant the same payment as the contractor's travel "
              'reimbursement?',
  'trap_records': ['A training grant of 800 dollars was approved for an external certification',
                   "A reimbursement of 800 dollars was approved for a contractor's travel"],
  'correct_answer_must_state': 'No stated connection: the amounts match by coincidence, and the '
                               'purposes differ.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: amount'},
 {'id': 'P8',
  'question': 'Was the background check vendor changed because the coding bootcamp was scheduled?',
  'trap_records': ['A background check vendor was changed after a contract review',
                   'A coding bootcamp was scheduled for junior engineers'],
  'correct_answer_must_state': 'No stated connection between a vendor change and a bootcamp; any '
                               'causal story is invented.',
  'expect_connected': False,
  'kind': 'Cross-category causal'}]

PUBLIC_POSITIVE = [{'id': 'PP1',
  'question': 'Was Employee E1034 recommended for promotion, and did the new salary appear in '
              'payroll?',
  'shared_identifier': 'Employee E1034',
  'shared_records': ['Employee E1034 was recommended for promotion by the committee',
                     "Employee E1034's new salary was reflected in the payroll run"],
  'correct_answer_must_state': 'Yes -- both records name Employee E1034.',
  'expect_connected': True},
 {'id': 'PP2',
  'question': 'Did Employee E1052 complete the data privacy course and also have a parental leave '
              'request approved?',
  'shared_identifier': 'Employee E1052',
  'shared_records': ['Employee E1052 completed the data privacy course before going on leave',
                     "Employee E1052's parental leave request was approved"],
  'correct_answer_must_state': 'Yes -- both records name Employee E1052.',
  'expect_connected': True},
 {'id': 'PP3',
  'question': 'Was Contractor K214 issued access before starting, and was their invoice later '
              'approved?',
  'shared_identifier': 'Contractor K214',
  'shared_records': ['Contractor K214 was issued a laptop and access card before starting',
                     "Contractor K214's invoice was approved after timesheet verification"],
  'correct_answer_must_state': 'Yes -- both records name Contractor K214.',
  'expect_connected': True}]

print(f"Public pack: {len(PUBLIC_ADVERSARIAL)} adversarial + {len(PUBLIC_POSITIVE)} positive-control questions")

In [ ]:
def structural_check(test):
    """No LLM needed: does the graph connect the two records this question is about?
    Returns (passed, connected, via)."""
    recs = test.get("trap_records") or test.get("shared_records")
    ia, ib = [f"R{rec_texts.index(t)}" for t in recs]
    chk = graph_check(ia, ib)
    connected = not chk["disconnected"]
    return connected == test["expect_connected"], connected, chk["via"]


def run_pack(pack, show_answers=True):
    """Runs baseline and grounded side by side on every question in a pack."""
    rows = []
    for t in pack:
        ok, connected, via = structural_check(t)
        print(f"[{t['id']}] {t['question']}")
        if show_answers:
            print("   Baseline :", baseline_rag_answer(t["question"])["answer"][:350].replace("\n", " "))
            print("   Grounded :", grounded_answer(t["question"])["answer"][:350].replace("\n", " "))
        print(f"   Graph    : {'CONNECTED' if connected else 'NOT connected'}"
              f"{' via ' + ', '.join(via) if via else ''} | expected "
              f"{'connected' if t['expect_connected'] else 'NOT connected'} -> {'PASS' if ok else 'FAIL'}")
        print("   Must say :", t["correct_answer_must_state"][:200])
        print()
        rows.append({"id": t["id"], "graph_pass": ok, "connected": connected})
    print(f"Graph verdict correct on {sum(r['graph_pass'] for r in rows)}/{len(rows)}")
    return rows

### Adversarial questions

In [ ]:
adv_rows = run_pack(PUBLIC_ADVERSARIAL)

### Positive controls: can your agent say *yes*?

In [ ]:
pos_rows = run_pack(PUBLIC_POSITIVE)

## 10. YOUR TASK: explain the outcomes

Fill in one line of `OUTCOMES` for each of the 11 public questions (P1-P8 and PP1-PP3). Each line has three fields. Read the **Baseline**, **Grounded**, **Graph** and **Must say** lines that the pack printed for that question, then decide:

| Field | The question it answers | How to decide |
| --- | --- | --- |
| `baseline_fabricated` | Did the plain model (no graph) **invent** something the records do not support? | Compare the **Baseline** answer with the **Must say** line. `True` only if the baseline claimed a link, a cause or "same person" that the records do not show. If it said "no", said "cannot tell", or said "yes" to a link that really exists, it did **not** fabricate: write `False`. |
| `grounded_correct` | Was **your** grounded answer right? | The **Graph** line ends in `PASS` -> `True`. `FAIL` -> `False`. |
| `explanation` | Why did it come out that way? | One sentence in terms of **your patterns**: which pattern matched (or did not), and which word was the shared ID, or a place, level, amount or period that is *not* an ID. |

**Most baselines are right on most questions.** `False` for `baseline_fabricated` is a normal, honest result; a long list of `True` that you did not actually see in the answers will lose marks. We score accurate, specific explanations, not the number of failures you found.

**How to write the explanation**

- Linked question (PP): *"Both records name <the ID>, and our <Name> pattern extracts it, so they are connected."*
- Not linked (P), shared word is a place, level, amount or period: *"Both mention <the word>, but that is a <place / amount / period>, a Kind 2 field reused by many events, not an event ID. We do not link on it."*
- Not linked (P), nothing shared: *"The two records name no shared ID, so no pattern links them."*
- Do not write "it might have..." or "it is not specified". That is guessing, and guessing is what the baseline does.

Example of the shape (not one of your questions):

```python
{"id": "P0", "baseline_fabricated": False, "grounded_correct": True,
 "explanation": "Both mention Shelf 3, but a shelf is a place, not an event ID, so no link."},
```

Then name one failure you still have in `REMAINING_FAILURE`. Be specific: a short honest note beats a vague claim.

In [ ]:
OUTCOMES = [
    # One dictionary per question, in this shape (see the table above):
    # {"id": "P1", "baseline_fabricated": ..., "grounded_correct": ..., "explanation": "..."},
    # Fill True or False for the first two fields; do not leave None.
    # Cover P1-P8 and PP1-PP3.
]

REMAINING_FAILURE = """
TODO: one case, or kind of case, where your system still gets it wrong (or could), and why.
"""


## 11. Submission summary

Run this last. This is what the judges read first.

In [ ]:
import json

def submission_summary():
    print("=== PATTERN REGISTER ===")
    show_register()
    # one machine-readable line the judges' tool reads: keep it
    print("REGISTRY_JSON:", json.dumps([{"name": p["name"], "regex": p["regex"]} for p in PATTERNS]))
    print("DATASET_ID:", DATASET_ID)
    print("\n=== OUTCOMES (public pack) ===")
    print(f"{'ID':<5} {'Baseline fabricated?':<22} {'Grounded correct?':<19} Explanation")
    for o in OUTCOMES:
        print(f"{o['id']:<5} {str(o['baseline_fabricated']):<22} {str(o['grounded_correct']):<19} {o['explanation']}")
    print("\n=== ONE FAILURE WE STILL HAVE ===")
    print(REMAINING_FAILURE)

submission_summary()

## 12. Before you submit

- Restart the runtime and **Run all**: it must work from a fresh state.
- `validate_registry()` reports no problems.
- `OUTCOMES` has an entry for every public question; `REMAINING_FAILURE` is filled in.
- Submit by the deadline announced on Day 1.